In [ ]:

# ===================================================
# ВЕРСИЯ 2. РУЧНАЯ РЕАЛИЗАЦИЯ MLP 
# ===================================================

class ManualStandardScaler:
    def fit(self, data):
        self.mean = np.mean(data, axis=0)
        self.std = np.std(data, axis=0)

        self.std[self.std == 0] = 1

    def transform(self, data):
        return (data - self.mean) / self.std

    def inverse_transform(self, data):
        return data * self.std + self.mean


class ManualMLPRegressor:
    def __init__(
        self,
        input_size,
        hidden1_size=64,
        hidden2_size=32,
        learning_rate=0.01,
        random_state=42
    ):
        np.random.seed(random_state)

        self.learning_rate = learning_rate

        # Веса между входным слоем и первым скрытым слоем
        self.W1 = np.random.randn(input_size, hidden1_size) * np.sqrt(2 / input_size)
        self.b1 = np.zeros((1, hidden1_size))

        # Веса между первым и вторым скрытым слоем
        self.W2 = np.random.randn(hidden1_size, hidden2_size) * np.sqrt(2 / hidden1_size)
        self.b2 = np.zeros((1, hidden2_size))

        # Веса между вторым скрытым слоем и выходом
        self.W3 = np.random.randn(hidden2_size, 1) * np.sqrt(2 / hidden2_size)
        self.b3 = np.zeros((1, 1))

    def relu(self, z):
        return np.maximum(0, z)

    def relu_derivative(self, z):
        return (z > 0).astype(float)

    def forward(self, X):
        # Прямое распространение

        self.z1 = X @ self.W1 + self.b1
        self.a1 = self.relu(self.z1)

        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = self.relu(self.z2)

        # Для регресси нв ыходе исп лин актив
        self.z3 = self.a2 @ self.W3 + self.b3

        return self.z3

    def backward(self, X, y, y_pred):
        
        # Обратное распространение ошибки

        m = X.shape[0]

        # Производная MSE по выходу 
        dz3 = 2 * (y_pred - y) / m

        dW3 = self.a2.T @ dz3
        db3 = np.sum(dz3, axis=0, keepdims=True)

        da2 = dz3 @ self.W3.T
        dz2 = da2 * self.relu_derivative(self.z2)

        dW2 = self.a1.T @ dz2
        db2 = np.sum(dz2, axis=0, keepdims=True)

        da1 = dz2 @ self.W2.T
        dz1 = da1 * self.relu_derivative(self.z1)

        dW1 = X.T @ dz1
        db1 = np.sum(dz1, axis=0, keepdims=True)

        # Обновление весов  градиентом
        self.W3 -= self.learning_rate * dW3
        self.b3 -= self.learning_rate * db3

        self.W2 -= self.learning_rate * dW2
        self.b2 -= self.learning_rate * db2

        self.W1 -= self.learning_rate * dW1
        self.b1 -= self.learning_rate * db1

    def fit(self, X, y, epochs=1000, batch_size=32):
        losses = []
        n_samples = X.shape[0]

        for epoch in range(epochs):
            indices = np.arange(n_samples)
            np.random.shuffle(indices)

            X_shuffled = X[indices]
            y_shuffled = y[indices]

            for start in range(0, n_samples, batch_size):
                end = start + batch_size

                X_batch = X_shuffled[start:end]
                y_batch = y_shuffled[start:end]

                y_pred_batch = self.forward(X_batch)
                self.backward(X_batch, y_batch, y_pred_batch)

            y_pred = self.forward(X)
            loss = np.mean((y_pred - y) ** 2)
            losses.append(loss)

            if epoch % 100 == 0:
                print(f"Epoch {epoch}, MSE: {loss:.6f}")

        return losses

    def predict(self, X):
        return self.forward(X)


def manual_mlp_regressor(df: pd.DataFrame):
    print_section("ВЕРСИЯ 2. Ручная реализация MLP-регрессора")

    df_manual = df.copy()

    if "Brand" in df_manual.columns:
        df_manual = pd.get_dummies(df_manual, columns=["Brand"], drop_first=False)

    X = df_manual.drop("Price", axis=1).values.astype(float)
    y = df_manual["Price"].values.reshape(-1, 1).astype(float)

    x_scaler = ManualStandardScaler()
    y_scaler = ManualStandardScaler()

    x_scaler.fit(X)
    y_scaler.fit(y)

    X_scaled = x_scaler.transform(X)
    y_scaled = y_scaler.transform(y)

    np.random.seed(42)

    indices = np.arange(X_scaled.shape[0])
    np.random.shuffle(indices)

    test_size = int(0.2 * X_scaled.shape[0])

    test_indices = indices[:test_size]
    train_indices = indices[test_size:]

    X_train = X_scaled[train_indices]
    y_train = y_scaled[train_indices]

    X_test = X_scaled[test_indices]
    y_test = y_scaled[test_indices]

    model = ManualMLPRegressor(
        input_size=X_train.shape[1],
        hidden1_size=64,
        hidden2_size=32,
        learning_rate=0.01,
        random_state=42
    )

    losses = model.fit(
        X_train,
        y_train,
        epochs=1000,
        batch_size=32
    )

    y_pred_scaled = model.predict(X_test)

    y_test_original = y_scaler.inverse_transform(y_test)
    y_pred_original = y_scaler.inverse_transform(y_pred_scaled)

    print_metrics(
        y_test_original,
        y_pred_original,
        "Метрики качества ручной модели"
    )

    result = pd.DataFrame({
        "Real Price": y_test_original.flatten(),
        "Predicted Price": y_pred_original.flatten()
    })

    print("\nПервые 10 предсказаний ручной модели:")
    print(result.head(10))

    plt.figure(figsize=(8, 5))
    plt.plot(losses)
    plt.xlabel("Эпоха")
    plt.ylabel("MSE")
    plt.title("График ошибки обучения ручного MLP")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    return model


df = pd.read_csv(PATH)

print_section("Загрузка обработанного датасета")
print("Shape:", df.shape)
print(df.head())
print(df.dtypes)



manual_model = manual_mlp_regressor(df)
